# 12. Fixed-capital-formation diagnostic

Quantify gross fixed capital formation relative to B.9 using an explicitly non-official diagnostic, in order to size public investment against the recorded balance.

**Reads**

- `outputs/tables/investment_diagnostic.csv`

**Writes**

- Nothing. The diagnostic table is persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 13

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. A diagnostic, not a balance definition

$$B^{before\ GFCF}_{i,t} = B_{i,t} + GFCF_{i,t}.$$

Adding gross fixed capital formation back to B.9 answers one narrow question: how
large is public investment compared with the recorded balance? It is **not** an
official fiscal indicator, and no fiscal rule in this repository is evaluated
against it.

In [ ]:
investment = pd.read_csv(TABLES / 'investment_diagnostic.csv')
diagnostic_columns = [
    'year',
    'balance_m_eur',
    'gfcf_m_eur',
    'balance_before_gfcf_m_eur',
    'gfcf_pct_gdp',
    'gfcf_share_abs_balance',
]
central = investment.loc[
    investment['sector'].eq('central_government') & investment['year'].ge(2015), diagnostic_columns
]
display(central.round(3))

In [ ]:
figure = figures.investment_diagnostic(investment, 'central_government')

## 2. Investment by sector

Regional and Local Government carries a large share of public investment relative
to its size, so the diagnostic behaves differently across subsectors.

In [ ]:
figure = figures.gfcf_by_sector(investment)

In [ ]:
display(
    investment.groupby('sector')[['gfcf_pct_gdp', 'gfcf_share_abs_balance']]
    .agg(['mean', 'median', 'max'])
    .round(3)
)

## Interpretation limits

1. The balance before GFCF is **not** a golden-rule balance, a structural
   balance, or any published indicator.
2. GFCF is **gross**: no consumption of fixed capital is netted off, so the
   diagnostic overstates the change in the public capital stock.
3. Investment is **lumpy**. Single-year ratios can be dominated by one project or
   one reclassification.

---

[Previous: 11. Primary balance and interest](11_primary_balance.ipynb) | [Next: 13. Debt and stock-flow reconciliation](13_debt_reconciliation.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```